# PostgreSQL Logical Backup, Restore, and Verification

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/CST4714_OER_Rebuild/notebooks/03_postgres_backup_restore.ipynb)

This notebook performs a real PostgreSQL logical backup and restores it into a
different database. The databases are temporary and isolated, so the lab teaches
the full evidence chain without risking a Supabase project.

**Evidence chain:** source checks -> dump artifact -> artifact inspection ->
separate restore -> structure checks -> data checks -> behavior check.

No cloud credential is required. The final section translates the same procedure
to Supabase without storing a connection URL.

## 1. Prepare PostgreSQL in the Notebook Runtime

Google Colab does not start with a PostgreSQL server, so the next cell installs
the free PostgreSQL package when it detects Colab and starts a local service. On a
computer where PostgreSQL is already running, it uses the current local server.

The command prefix is shown explicitly. It changes only because Colab's local
server is owned by its `postgres` operating-system user.

In [1]:
import hashlib
import os
from pathlib import Path
import re
import shutil
import subprocess

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ

if IN_COLAB and shutil.which("pg_dump") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(
        ["apt-get", "-qq", "install", "-y", "postgresql", "postgresql-client"],
        check=True,
    )

if IN_COLAB:
    subprocess.run(["service", "postgresql", "start"], check=True)

PG_PREFIX = ["sudo", "-u", "postgres"] if IN_COLAB else []
print("Running in Colab:", IN_COLAB)
print("PostgreSQL command prefix:", PG_PREFIX or "current local user")

readiness = subprocess.run(
    PG_PREFIX + ["pg_isready"],
    check=True,
    text=True,
    capture_output=True,
)
print(readiness.stdout.strip())

server_version_result = subprocess.run(
    PG_PREFIX + ["psql", "--tuples-only", "--no-align", "--command", "SHOW server_version_num"],
    check=True,
    text=True,
    capture_output=True,
)
server_version_num = int(server_version_result.stdout.strip())
server_major = server_version_num // 10000

candidate_directories = []
path_pg_dump = shutil.which("pg_dump")
if path_pg_dump:
    candidate_directories.append(Path(path_pg_dump).parent)
candidate_directories.extend(
    [
        Path(f"/Applications/Postgres.app/Contents/Versions/{server_major}/bin"),
        Path(f"/usr/lib/postgresql/{server_major}/bin"),
    ]
)

PG_BIN = None
for candidate_directory in candidate_directories:
    candidate_dump = candidate_directory / "pg_dump"
    if not candidate_dump.exists():
        continue
    version_text = subprocess.run(
        [str(candidate_dump), "--version"],
        check=True,
        text=True,
        capture_output=True,
    ).stdout
    version_match = re.search(r"(\d+)(?:\.\d+)?", version_text)
    if version_match and int(version_match.group(1)) >= server_major:
        PG_BIN = candidate_directory
        break

if PG_BIN is None:
    raise RuntimeError(
        f"PostgreSQL server major version {server_major} needs pg_dump {server_major} "
        "or newer. Install a compatible PostgreSQL client and rerun this cell."
    )

PSQL = str(PG_BIN / "psql")
CREATEDB = str(PG_BIN / "createdb")
DROPDB = str(PG_BIN / "dropdb")
PG_DUMP = str(PG_BIN / "pg_dump")
PG_RESTORE = str(PG_BIN / "pg_restore")

print("Server major version:", server_major)
print("Compatible client directory:", PG_BIN)
print(subprocess.run([PG_DUMP, "--version"], check=True, text=True, capture_output=True).stdout.strip())

Running in Colab: False
PostgreSQL command prefix: current local user
/tmp:5432 - accepting connections
Server major version: 18
Compatible client directory: /Applications/Postgres.app/Contents/Versions/18/bin
pg_dump (PostgreSQL) 18.4 (Postgres.app)


## 2. Create a Source Database

The source and restore databases have visibly different names. The setup contains
keys, relationships, a status constraint, and a few rows so later checks can test
more than counts.

In [2]:
SOURCE_DB = "cst4714_recovery_source"
RESTORE_DB = "cst4714_recovery_restore"
SETUP_FILE = Path("/tmp/cst4714_metro_support_setup.sql")
DUMP_FILE = Path("/tmp/cst4714_metro_support.dump")

setup_sql = """
CREATE SCHEMA metro_support;

CREATE TABLE metro_support.users (
    user_id integer PRIMARY KEY,
    display_name text NOT NULL,
    role text NOT NULL
);

CREATE TABLE metro_support.tickets (
    ticket_id integer PRIMARY KEY,
    requester_id integer NOT NULL REFERENCES metro_support.users(user_id),
    status text NOT NULL CONSTRAINT tickets_status_allowed
        CHECK (status IN ('new', 'open', 'in_progress', 'resolved', 'closed')),
    subject text NOT NULL
);

CREATE TABLE metro_support.ticket_events (
    event_id integer PRIMARY KEY,
    ticket_id integer NOT NULL REFERENCES metro_support.tickets(ticket_id),
    event_type text NOT NULL
);

INSERT INTO metro_support.users VALUES
    (101, 'Maya Chen', 'resident'),
    (102, 'Luis Rivera', 'resident'),
    (201, 'Priya Shah', 'agent');

INSERT INTO metro_support.tickets VALUES
    (1001, 101, 'open', 'Streetlight dark near bus stop'),
    (1002, 102, 'in_progress', 'Missed recycling pickup'),
    (1003, 101, 'resolved', 'Low water pressure');

INSERT INTO metro_support.ticket_events VALUES
    (5001, 1001, 'created'),
    (5002, 1001, 'assigned'),
    (5003, 1002, 'created'),
    (5004, 1003, 'created'),
    (5005, 1003, 'status_changed');
"""

SETUP_FILE.write_text(setup_sql, encoding="utf-8")

for database_name in (RESTORE_DB, SOURCE_DB):
    subprocess.run(
        PG_PREFIX + [DROPDB, "--if-exists", database_name],
        check=True,
        text=True,
        capture_output=True,
    )

subprocess.run(PG_PREFIX + [CREATEDB, SOURCE_DB], check=True)
subprocess.run(
    PG_PREFIX
    + [PSQL, "--set=ON_ERROR_STOP=on", "--dbname", SOURCE_DB, "--file", str(SETUP_FILE)],
    check=True,
    text=True,
    capture_output=True,
)
print("Created source database:", SOURCE_DB)

Created source database: cst4714_recovery_source


## 3. Record the Source State

Expected source counts are 3 users, 3 tickets, and 5 events. We also inspect the
named status constraint. These checks become the baseline for the restored target.

In [3]:
source_check_sql = """
SELECT 'users=' || count(*) FROM metro_support.users;
SELECT 'tickets=' || count(*) FROM metro_support.tickets;
SELECT 'ticket_events=' || count(*) FROM metro_support.ticket_events;
SELECT 'constraint=' || conname
FROM pg_constraint
WHERE conname = 'tickets_status_allowed';
"""

source_check = subprocess.run(
    PG_PREFIX + [PSQL, "--tuples-only", "--no-align", "--dbname", SOURCE_DB, "--command", source_check_sql],
    check=True,
    text=True,
    capture_output=True,
)
print(source_check.stdout.strip())

users=3
tickets=3
ticket_events=5
constraint=tickets_status_allowed


## 4. Create the Logical Backup Artifact

`pg_dump --format=custom` creates an archive for `pg_restore`. `--no-owner` and
`--no-privileges` make the classroom restore less dependent on identical roles,
but they also mean ownership and grants need a separate recovery plan.

In [4]:
if DUMP_FILE.exists():
    DUMP_FILE.unlink()

subprocess.run(
    PG_PREFIX
    + [
        PG_DUMP,
        "--format=custom",
        "--schema=metro_support",
        "--no-owner",
        "--no-privileges",
        "--file",
        str(DUMP_FILE),
        SOURCE_DB,
    ],
    check=True,
)

dump_bytes = DUMP_FILE.read_bytes()
dump_sha256 = hashlib.sha256(dump_bytes).hexdigest()
print("Dump path:", DUMP_FILE)
print("Dump size in bytes:", len(dump_bytes))
print("SHA-256:", dump_sha256)

Dump path: /tmp/cst4714_metro_support.dump
Dump size in bytes: 5245
SHA-256: 850b9ef294409d5c96cdf20455d02de172fd36d1a499305b78ec8b0b6afcb0b1


## 5. Inspect the Artifact Before Restoring

`pg_restore --list` reads the archive table of contents. Seeing the expected
schema, tables, data, constraints, and indexes is useful evidence, but it still
does not prove that restoration succeeds.

In [5]:
archive_list = subprocess.run(
    PG_PREFIX + [PG_RESTORE, "--list", str(DUMP_FILE)],
    check=True,
    text=True,
    capture_output=True,
)

important_lines = [
    line
    for line in archive_list.stdout.splitlines()
    if any(term in line for term in ("SCHEMA", "TABLE ", "TABLE DATA", "CONSTRAINT", "INDEX"))
]
print("\n".join(important_lines))

6; 2615 17090 SCHEMA - metro_support atiliobarreda
222; 1259 17118 TABLE metro_support ticket_events atiliobarreda
221; 1259 17101 TABLE metro_support tickets atiliobarreda
220; 1259 17091 TABLE metro_support users atiliobarreda
3836; 0 17118 TABLE DATA metro_support ticket_events atiliobarreda
3835; 0 17101 TABLE DATA metro_support tickets atiliobarreda
3834; 0 17091 TABLE DATA metro_support users atiliobarreda
3684; 2606 17127 CONSTRAINT metro_support ticket_events ticket_events_pkey atiliobarreda
3682; 2606 17112 CONSTRAINT metro_support tickets tickets_pkey atiliobarreda
3680; 2606 17100 CONSTRAINT metro_support users users_pkey atiliobarreda
3686; 2606 17128 FK CONSTRAINT metro_support ticket_events ticket_events_ticket_id_fkey atiliobarreda
3685; 2606 17113 FK CONSTRAINT metro_support tickets tickets_requester_id_fkey atiliobarreda


## 6. Restore Into a Different Database

The destination is empty and separate. `--exit-on-error` prevents an archive with
an early failure from looking successful merely because later items continued.

In [6]:
subprocess.run(PG_PREFIX + [CREATEDB, RESTORE_DB], check=True)

restore_result = subprocess.run(
    PG_PREFIX
    + [
        PG_RESTORE,
        "--exit-on-error",
        "--no-owner",
        "--no-privileges",
        "--dbname",
        RESTORE_DB,
        str(DUMP_FILE),
    ],
    check=True,
    text=True,
    capture_output=True,
)

print("Restored into separate database:", RESTORE_DB)
print("Restore exit code:", restore_result.returncode)

Restored into separate database: cst4714_recovery_restore
Restore exit code: 0


## 7. Verify Structure, Data, and Relationships

The following checks ask different questions:

- Do all three tables exist?
- Do row counts match the source baseline?
- Are there tickets with a missing requester relationship?
- Does a meaningful report return the expected grouped result?

In [7]:
restore_check_sql = """
SELECT 'tables=' || count(*)
FROM information_schema.tables
WHERE table_schema = 'metro_support' AND table_type = 'BASE TABLE';

SELECT 'users=' || count(*) FROM metro_support.users;
SELECT 'tickets=' || count(*) FROM metro_support.tickets;
SELECT 'ticket_events=' || count(*) FROM metro_support.ticket_events;

SELECT 'orphan_tickets=' || count(*)
FROM metro_support.tickets AS t
LEFT JOIN metro_support.users AS u ON u.user_id = t.requester_id
WHERE u.user_id IS NULL;

SELECT status || '=' || count(*)
FROM metro_support.tickets
GROUP BY status
ORDER BY status;
"""

restore_check = subprocess.run(
    PG_PREFIX + [PSQL, "--tuples-only", "--no-align", "--dbname", RESTORE_DB, "--command", restore_check_sql],
    check=True,
    text=True,
    capture_output=True,
)
print(restore_check.stdout.strip())

tables=3
users=3
tickets=3
ticket_events=5
orphan_tickets=0
in_progress=1
open=1
resolved=1


## 8. Verify Behavior With an Expected Failure

A restored table can contain rows while missing an integrity rule. This insert
must fail because `almost_done` is not an allowed status. We treat the nonzero
command result as expected evidence and print only the final error line.

In [8]:
invalid_insert = subprocess.run(
    PG_PREFIX
    + [
        PSQL,
        "--set=ON_ERROR_STOP=on",
        "--dbname",
        RESTORE_DB,
        "--command",
        """
        INSERT INTO metro_support.tickets
            (ticket_id, requester_id, status, subject)
        VALUES
            (1099, 101, 'almost_done', 'Constraint restore test');
        """,
    ],
    check=False,
    text=True,
    capture_output=True,
)

print("Expected nonzero exit code:", invalid_insert.returncode)
error_lines = [line for line in invalid_insert.stderr.splitlines() if line.strip()]
print("Expected constraint evidence:", error_lines[-1] if error_lines else "no error text")
assert invalid_insert.returncode != 0, "The restored status constraint did not reject invalid data."

Expected nonzero exit code: 1
Expected constraint evidence: DETAIL:  Failing row contains (1099, 101, almost_done, Constraint restore test).


## 9. Compare and Clean Up

The source and restore evidence should agree on required counts. Cleanup removes
the temporary databases only after all verification has finished. The dump file
remains in `/tmp` until the notebook runtime ends.

In [9]:
assert "users=3" in source_check.stdout and "users=3" in restore_check.stdout
assert "tickets=3" in source_check.stdout and "tickets=3" in restore_check.stdout
assert "ticket_events=5" in source_check.stdout and "ticket_events=5" in restore_check.stdout
assert "orphan_tickets=0" in restore_check.stdout

print("Required source and restore checks agree.")

for database_name in (RESTORE_DB, SOURCE_DB):
    subprocess.run(
        PG_PREFIX + [DROPDB, "--if-exists", database_name],
        check=True,
        text=True,
        capture_output=True,
    )

print("Removed temporary source and restore databases.")

Required source and restore checks agree.


Removed temporary source and restore databases.


## 10. Translate the Procedure to Supabase

Supabase Free projects do not receive the automatic database backups described
for paid plans. A course recovery plan therefore uses a logical connection and
runtime credential, for example:

```bash
pg_dump --format=custom --no-owner --no-privileges   --file=project.dump "$DATABASE_URL"

pg_restore --exit-on-error --no-owner --no-privileges   --dbname="$RESTORE_DATABASE_URL" project.dump
```

Use the current Supabase connection guidance. A direct endpoint may require IPv6;
the session pooler offers an IPv4-compatible path in many networks. The source and
restore URLs must point to different, approved targets. Enter URLs at runtime,
never in the notebook.

Official free resources:

- <https://supabase.com/docs/guides/platform/backups>
- <https://supabase.com/docs/guides/database/connecting-to-postgres>
- <https://www.postgresql.org/docs/current/backup-dump.html>

## Recovery Record: Complete Before Submission

**Failure scope:** [what this artifact is intended to recover]

**Artifact:** [format, file name, size, and abbreviated checksum]

**Safety boundary:** [how the restore target differs from the source]

**Five checks:** [two structure, two data/relationship, and one behavior check]

**What the checks do not prove:** [one limitation]

**Supabase translation:** [which connection path you would use and how you would
enter the source and restore URLs without saving them]

**RPO/RTO implication:** [what backup frequency and restore practice would be
needed for a stated requirement]

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.